<a href="https://colab.research.google.com/github/Innovatewithapple/RNNProjects/blob/main/RNNBasic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=3d6b539b6f71fe77c46c0328b2b54d4ed9e1b26174cf01a68155fc823671151a
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [2]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout, Input,Bidirectional,LSTM
import tensorflow_datasets as tfds
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
from lime.lime_text import LimeTextExplainer

In [3]:
# 1. Download the IMDB dataset
# 'as_supervised=True' gives us (text, label) pairs directly
dataset, info = tfds.load('imdb_reviews', with_info=True, as_supervised=True)

# 2. Split into Train and Validation (Standard 80/20 split)
train_data = dataset['train'].batch(32).prefetch(tf.data.AUTOTUNE)
test_data = dataset['test'].batch(32).prefetch(tf.data.AUTOTUNE)

# 3. Get just the sentences for the .adapt() step
# We take a sample of the text to build our vocabulary
train_sentences = dataset['train'].map(lambda x, y: x)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.SZMRKL_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.SZMRKL_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.SZMRKL_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


In [4]:
# 1. Grab 3 examples from the 'train' split
for text, label in dataset['train'].take(3):
    print(f"SENTIMENT: {'Positive' if label == 1 else 'Negative'}")
    print(f"REVIEW: {text.numpy().decode('utf-8')[:250]}...") # Show first 250 characters
    print("-" * 50)

SENTIMENT: Negative
REVIEW: This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline...
--------------------------------------------------
SENTIMENT: Negative
REVIEW: I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell asleep because the film wa...
--------------------------------------------------
SENTIMENT: Negative
REVIEW: Mann photographs the Alberta Rocky Mountains in a superb fashion, and Jimmy Stewart and Walter Brennan give enjoyable performances as they always seem to do. <br /><br />But come on Hollywood - a Mountie telling the people of Dawson City, Yukon to el...
--------------------------------------------------


In [ ]:
# 1. Define the "Resize" for text (e.g., only look at the first 100 words)
max_words = 10000 # Top 10,000 most common words
max_len = 100     # "Resize" every review to 100 words

vectorize_layer = TextVectorization(
    max_tokens=max_words,
    output_mode='int',
    output_sequence_length=max_len
)

# 2. Adapt to your training text (like finding the 'mean' for normalization)
vectorize_layer.adapt(train_sentences)

In [6]:
#SimpleRNN
model_rnn = Sequential([
    Input(shape=(1,), dtype=tf.string), # Input is a raw string
    vectorize_layer,                   # Step 1: Turn string to numbers

    # Step 2: The "Feature Finder" (turns IDs into 64-number vectors)
    Embedding(input_dim=max_words, output_dim=128),

    # Step 3: The "Memory" (The RNN Brick)
    # This is the equivalent of your Conv2D/MaxPool layers!
    SimpleRNN(128),

    # Step 4: The "Head" (Exactly like your CNN!)
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid') # Binary verdict: Positive or Negative
])

In [ ]:
# 1. dropout=0.2 (The "Input" Shield)This works on the data entering the layer (the 128 features from the Embedding).The Logic: It randomly "hides" 20% of the words or features in the current sentence.The Goal: It forces the model to stop being obsessed with specific "trigger" words (like always looking for the word "Excellent") and instead learn the whole context of the sentence. It makes the model's "eyes" stronger.2. recurrent_dropout=0.2 (The "Memory" Shield)This is the "Professional" one. It works on the Hidden State (the memory) that loops from one word to the next.The Logic: As the model moves from Word #1 to Word #2, it randomly "forgets" 20% of its internal memory.The Goal: It prevents the model from memorizing specific patterns of reviews. If it can't rely on a perfect memory of the past, it’s forced to learn the "vibe" or "sentiment" of the sequence instead of just "copy-pasting" the data.

In [13]:
#LSTM
# --- STEP 3: THE SKYSCRAPER (Model Build) ---
model_bilstm = Sequential([
    Input(shape=(1,), dtype=tf.string), # One raw string package per example
    vectorize_layer,                   # String -> Integer IDs

    # THE EYES: 128 features per word
    Embedding(input_dim=max_words, output_dim=128),

    # THE BRAIN: Bidirectional LSTM with 64 memory units
    # 'recurrent_dropout' protects the memory between steps
    LSTM(64, dropout=0.2, recurrent_dropout=0.2), # here the first dropout use so that model cannot focus on particular words too much, and second dropout
    #Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),

    # THE JUDGE: Final decision logic
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid') # Binary answer (0=Neg, 1=Pos)
])

In [8]:
early_stop = EarlyStopping(
    monitor='val_loss',     # 1. Watch the validation loss
    patience=3,              # 2. Wait 3 epochs before giving up
    restore_best_weights=True, # 3. Keep the version that was the 'smartest'
    verbose=1                # 4. Tell us when it stops!
)

In [9]:
model = model_bilstm

In [14]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# It works exactly like your CNN .fit()!
model.fit(
    train_data,
    validation_data=test_data,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 129s 160ms/step - accuracy: 0.8656 - loss: 0.3436 - val_accuracy: 0.7918 - val_loss: 0.4723
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 125s 160ms/step - accuracy: 0.8946 - loss: 0.2768 - val_accuracy: 0.8037 - val_loss: 0.4840
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 123s 157ms/step - accuracy: 0.9188 - loss: 0.2212 - val_accuracy: 0.7899 - val_loss: 0.5990
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.


In [11]:
# Create a list of the actual sentences from your validation data
test_texts = []
test_labels = []

# Take the first 100 reviews to have a "pool" to test from
for text, label in test_data.unbatch().take(100):
    test_texts.append(text.numpy().decode('utf-8'))
    test_labels.append(label.numpy())

print(f"✅ Created a list of {len(test_texts)} reviews for LIME to test!")

✅ Created a list of 100 reviews for LIME to test!
